# TriageRL — Agent Training (Colab)

Trains the DQN and action-masked PPO triage agents in the simulated ED environment and compares both against rule-based baselines with confidence intervals and paired significance tests.

**Before running:** upload the `triagerl` project folder (zip it, upload, unzip), or clone from your GitHub repo. If you already have a trained DQN, upload `dqn_triage.zip` into `triagerl/models/` to skip retraining it.

In [ ]:
# Option A: upload triagerl.zip via the Files panel, then:
# !unzip -o triagerl.zip
# Option B: clone from GitHub:
# !git clone https://github.com/<your-username>/triagerl.git
%cd triagerl
!pip -q install gymnasium stable-baselines3 sb3-contrib scipy pandas matplotlib tqdm rich

In [ ]:
# Sanity check: environment + reward config + action masking
from triagerl.ed_env import EDTriageEnv, HOLD_COST
from triagerl.baselines import severity_policy

print('HOLD_COST:', HOLD_COST)
env = EDTriageEnv(seed=0)
env.reset(seed=0)
print('action mask:', env.action_masks())
done = False
while not done:
    _, _, term, trunc, _ = env.step(severity_policy(env))
    done = term or trunc
env.episode_metrics()

## Train DQN (skip if you already have models/dqn_triage.zip)

In [ ]:
from triagerl.train import train
model = train(timesteps=300_000, out="models/dqn_triage")

## Train MaskablePPO (action-masked agent)

In [ ]:
from triagerl.train_ppo import train as train_ppo
ppo = train_ppo(timesteps=300_000, out="models/ppo_triage")

## Evaluate: both agents vs baselines

Outputs `summary.csv`, `summary_ci.csv` (95% CIs), `significance.csv` (paired Wilcoxon tests), and `comparison.png` (with CI error bars). Tip: also evaluate the best checkpoints (`models/best_model` for DQN, `models/ppo_best/best_model` for PPO) and keep whichever wins.

In [ ]:
from triagerl.evaluate import main
summary = main(model="models/dqn_triage", episodes=30, out_dir="results",
               ppo_model="models/ppo_triage")
summary

In [ ]:
from IPython.display import Image
Image("results/comparison.png")

In [ ]:
# Download models + results
from google.colab import files
files.download("models/ppo_triage.zip")
files.download("models/ppo_best/best_model.zip")
files.download("results/summary_ci.csv")
files.download("results/significance.csv")
files.download("results/comparison.png")